In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

train    = pd.read_parquet(DATA_DIR / "train_v2.parquet")
txn      = pd.read_parquet(DATA_DIR / "txn_features.parquet")
members  = pd.read_parquet(DATA_DIR / "members_features.parquet")
logs     = pd.read_parquet(DATA_DIR / "logs_features.parquet")

print(f"train:   {train.shape}")
print(f"txn:     {txn.shape}")
print(f"members: {members.shape}")
print(f"logs:    {logs.shape}")

print(f"\ntrain columns:  {list(train.columns)}")

train:   (970960, 2)
txn:     (970960, 38)
members: (860967, 12)
logs:    (855160, 45)

train columns:  ['msno', 'is_churn']


In [ ]:
master = train.copy()

master = master.merge(txn,     on='msno', how='left')
master = master.merge(members, on='msno', how='left')
master = master.merge(logs,    on='msno', how='left')

print(f"Master shape: {master.shape}")
print(f"\nChurn rate preserved: {master['is_churn'].mean():.4f} (should be 0.0899)")
print(f"\nUsers missing txn:     {master['current_auto_renew'].isnull().sum():,}")
print(f"Users missing members: {master['city'].isnull().sum():,}")
print(f"Users missing logs:    {master['days_since_last_listen'].isnull().sum():,}")

Master shape: (970960, 94)

Churn rate preserved: 0.0899 (should be 0.0899)

Users missing txn:     0
Users missing members: 109,993
Users missing logs:    115,800


In [ ]:
# last_churn flags (January churn labels)
# Check if January labels exist
import os

# The competition provided train.csv (Jan labels) and train_v2.csv (Feb labels)
# train_v2 = February prediction target (what we're predicting)
# train = January labels (used as a feature)

jan_labels_path = DATA_DIR / "train_v2.parquet"

print("Parquet files in data dir:")
for f in sorted(DATA_DIR.glob("*.parquet")):
    size_mb = os.path.getsize(f) / 1024**2
    print(f"  {f.name}: {size_mb:.1f} MB")

Parquet files in data dir:
  logs_all_clean.parquet: 4160.0 MB
  logs_features.parquet: 146.8 MB
  members_features.parquet: 40.9 MB
  members_v3.parquet: 225.2 MB
  train_v2.parquet: 30.8 MB
  transactions.parquet: 726.7 MB
  transactions_v2.parquet: 50.3 MB
  txn_features.parquet: 61.9 MB
  user_logs.parquet: 6296.2 MB
  user_logs_v2.parquet: 732.6 MB


In [ ]:
#Check for January labels CSV
DATA_ROOT = Path.home() / "kkbox-churn" / "data"

jan_csv = DATA_ROOT / "train.csv"
print(f"train.csv exists: {jan_csv.exists()}")

if jan_csv.exists():
    import os
    size_mb = os.path.getsize(jan_csv) / 1024**2
    print(f"Size: {size_mb:.1f} MB")

train.csv exists: True
Size: 44.5 MB


In [ ]:
jan_labels = pd.read_csv(DATA_ROOT / "train.csv")
print(f"January labels shape: {jan_labels.shape}")
print(f"Columns: {list(jan_labels.columns)}")
print(f"Churn rate: {jan_labels['is_churn'].mean():.4f}")
print(f"Sample:\n{jan_labels.head(3).to_string()}")

January labels shape: (992931, 2)
Columns: ['msno', 'is_churn']
Churn rate: 0.0639
Sample:
                                           msno  is_churn
0  waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=         1
1  QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=         1
2  fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=         1


In [ ]:
jan_labels = jan_labels.rename(columns={'is_churn': 'jan_is_churn'})

# Merge January labels onto master
master = master.merge(jan_labels, on='msno', how='left')

# Create flags
master['last_churn']     = (master['jan_is_churn'] == 1).astype(int)
master['last_not_churn'] = (master['jan_is_churn'] == 0).astype(int)
master['last_not_in']    = master['jan_is_churn'].isnull().astype(int)

# Drop the raw column
master.drop(columns=['jan_is_churn'], inplace=True)

# Verify — these three flags must sum to 1 for every user
check = master[['last_churn','last_not_churn','last_not_in']].sum(axis=1)
print(f"Flag sum check (all should be 1): {check.unique()}")

print(f"\nlast_churn:     {master['last_churn'].sum():,}  ({master['last_churn'].mean():.4f})")
print(f"last_not_churn: {master['last_not_churn'].sum():,}  ({master['last_not_churn'].mean():.4f})")
print(f"last_not_in:    {master['last_not_in'].sum():,}  ({master['last_not_in'].mean():.4f})")

print(f"\nMaster shape: {master.shape}")

Flag sum check (all should be 1): [1]

last_churn:     16,321  (0.0168)
last_not_churn: 865,380  (0.8913)
last_not_in:    89,259  (0.0919)

Master shape: (970960, 97)


In [ ]:
# Cell 6 — Cross-table features 

# 1. has_listening_history
master['has_listening_history'] = master['days_since_last_listen'].notna().astype(int)

# 2. days_between_last_listen_and_expiry — use raw dates directly
master['days_between_last_listen_and_expiry'] = (
    (master['last_expire_date'] - master['last_listen_date']).dt.days
)

# 3. listening_per_dollar
master['listening_per_dollar'] = (
    master['secs_all'] / master['total_amount_paid'].replace(0, np.nan)
)

# 4. is_extreme_risk
master['is_extreme_risk'] = (
    (master['stopped_listening'] == 1) &
    (master['total_cancellations'] > 0)
).astype(int)

# 5. is_silent_loyal
master['is_silent_loyal'] = (
    (master['has_listening_history'] == 0) &
    (master['total_cancellations'] == 0)
).astype(int)

# 6. auto_off_inactive
master['auto_off_inactive'] = (
    (master['current_auto_renew'] == 0) &
    (master['days_since_last_listen'] >= 30)
).astype(int)

# 7. listen_to_plan_ratio
master['listen_to_plan_ratio'] = (
    master['secs_all'] / master['customer_tenure_days'].replace(0, np.nan)
)

# 8. active_days_per_sub_day
master['active_days_per_sub_day'] = (
    master['active_days_all'] / master['customer_tenure_days'].replace(0, np.nan)
)

print(f"has_listening_history:                     {master['has_listening_history'].sum():,}")
print(f"is_extreme_risk:                           {master['is_extreme_risk'].sum():,}")
print(f"is_silent_loyal:                           {master['is_silent_loyal'].sum():,}")
print(f"auto_off_inactive:                         {master['auto_off_inactive'].sum():,}")
print(f"days_between_last_listen_and_expiry nulls: {master['days_between_last_listen_and_expiry'].isnull().sum():,}")
print(f"listening_per_dollar nulls:                {master['listening_per_dollar'].isnull().sum():,}")
print(f"listen_to_plan_ratio nulls:                {master['listen_to_plan_ratio'].isnull().sum():,}")
print(f"active_days_per_sub_day nulls:             {master['active_days_per_sub_day'].isnull().sum():,}")

print(f"\nMaster shape: {master.shape}")

has_listening_history:                     855,160
is_extreme_risk:                           7,177
is_silent_loyal:                           104,140
auto_off_inactive:                         3,893
days_between_last_listen_and_expiry nulls: 115,800
listening_per_dollar nulls:                116,181
listen_to_plan_ratio nulls:                115,807
active_days_per_sub_day nulls:             115,807

Master shape: (970960, 105)


In [ ]:
# Binary flags: fill with 0 for users missing from members/logs
binary_cols = [
    'is_city_1', 'age_known', 'gender_known', 'unknown_gender_young',
    'is_invalid_registration', 'has_listening_15d', 'has_listening_30d',
    'stopped_listening', 'is_extreme_risk', 'is_silent_loyal',
    'auto_off_inactive', 'has_listening_history',
    'last_churn', 'last_not_churn', 'last_not_in',
    'discount_flag', 'has_long_plan', 'has_price_increase',
    'payment_method_changed', 'switched_off_flag', 'cancelled_last_30_days'
]

for col in binary_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0).astype(int)

# days_since_last_cancel: fill with large sentinel (never cancelled)
master['days_since_last_cancel'] = master['days_since_last_cancel'].fillna(9999)

# t2_amount: fill with t1_amount (only 1 transaction)
master['t2_amount'] = master['t2_amount'].fillna(master['t1_amount'])

# Verify no unexpected nulls in critical features
critical_cols = [
    'current_auto_renew', 'days_since_last_transaction', 'days_to_expiry',
    'total_transactions', 'last_churn', 'has_listening_history'
]
print("Null check on critical columns:")
for col in critical_cols:
    print(f"  {col}: {master[col].isnull().sum():,}")

print(f"\nTotal nulls across all columns: {master.isnull().sum().sum():,}")

print(f"Master shape: {master.shape}")

Null check on critical columns:
  current_auto_renew: 0
  days_since_last_transaction: 0
  days_to_expiry: 0
  total_transactions: 0
  last_churn: 0
  has_listening_history: 0

Total nulls across all columns: 7,574,027
Master shape: (970960, 105)


In [9]:
# Cell 8 — Save master dataset
out_path = DATA_DIR / "master_dataset.parquet"
master.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Saved: {out_path}")
print(f"Size: {size_mb:.1f} MB")
print(f"Shape verify: {pd.read_parquet(out_path).shape}")
print(f"\nChurn rate: {master['is_churn'].mean():.4f}")
print(f"Columns: {len(master.columns)}")
print(f"\nColumn list:\n{list(master.columns)}")

Saved: /Users/harshithnr/kkbox-churn/data/parquet/master_dataset.parquet
Size: 201.9 MB
Shape verify: (970960, 105)

Churn rate: 0.0899
Columns: 105

Column list:
['msno', 'is_churn', 'last_transaction_date', 'last_expire_date', 'current_auto_renew', 'current_is_cancel', 'current_plan_days', 'current_payment_method', 'days_since_last_transaction', 'days_since_last_cancel', 'customer_tenure_days', 'days_to_expiry', 'total_transactions', 'total_cancellations', 'auto_renew_rate', 'cancel_rate', 'total_amount_paid', 'avg_amount_paid', 'pay_per_day', 'avg_plan_days', 'max_plan_days', 'has_long_plan', 'payment_methods_used', 'total_discount', 'discount_flag', 'cancelled_last_30_days', 'cancel_last3', 'autorenew_last3', 'cancel_last5', 'autorenew_last5', 'price_change', 'has_price_increase', 'payment_method_changed', 'auto_renew_switch', 'switched_off_flag', 'avg_interval_days', 'nopay_ratio', 't1_amount', 't2_amount', 'city', 'registered_via', 'bd_clean', 'age_known', 'gender_known', 'unknow

In [10]:
# Cell 9 — Quick audit
print('auto_not_cancel_ratio' in master.columns)
print('plan_changes' in master.columns)

False
False


In [11]:
print('auto_not_cancel_ratio' in txn.columns)
print('plan_changes' in txn.columns)
print(txn.shape)

False
False
(970960, 38)


In [12]:
txn_check = pd.read_parquet(DATA_DIR / "txn_features.parquet")
print(txn_check.shape)
print('auto_not_cancel_ratio' in txn_check.columns)
print('plan_changes' in txn_check.columns)

(970960, 38)
False
False


In [13]:
# Cell 9 — Add missing txn patch features
import duckdb

DATA_DIR_RAW = Path.home() / "kkbox-churn" / "data" / "parquet"

con = duckdb.connect()
con.execute("SET threads=2")
con.execute("SET memory_limit='10GB'")
con.execute(f"CREATE OR REPLACE VIEW txn_all AS SELECT * FROM '{DATA_DIR_RAW}/transactions.parquet' UNION ALL SELECT * FROM '{DATA_DIR_RAW}/transactions_v2.parquet'")
con.execute(f"CREATE OR REPLACE VIEW train AS SELECT * FROM '{DATA_DIR_RAW}/train_v2.parquet'")

patch_missing = con.execute("""
SELECT
    msno,
    AVG(CASE WHEN is_auto_renew = 1 AND is_cancel = 0 THEN 1.0 ELSE 0.0 END) AS auto_not_cancel_ratio,
    COUNT(DISTINCT payment_plan_days)                                           AS plan_changes
FROM txn_all
WHERE msno IN (SELECT msno FROM train)
GROUP BY msno
""").df()

print(f"Patch shape: {patch_missing.shape}")
print(patch_missing.head(3).to_string())

Patch shape: (970960, 3)
                                           msno  auto_not_cancel_ratio  plan_changes
0  GCvXuOi6qjWcMevstArf5GCQVRGJgSIg26J5UxFoqeM=                    1.0             2
1  6Cw+F8uxLXprS0iELlFMQlXsGyeC28yWzqr116eARfU=                    1.0             2
2  yEEfWA0OzLjUd+e6x3tA6Yz2gJPQ7y2eYzkmUCLjk8g=                    1.0             2


In [14]:
# Cell 10 — Merge missing features into master and resave
master = master.merge(patch_missing, on='msno', how='left')

print(f"Master shape: {master.shape}")
print(f"auto_not_cancel_ratio nulls: {master['auto_not_cancel_ratio'].isnull().sum()}")
print(f"plan_changes nulls: {master['plan_changes'].isnull().sum()}")

out_path = DATA_DIR / "master_dataset.parquet"
master.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Saved: {out_path}")
print(f"Size: {size_mb:.1f} MB")
print(f"Shape verify: {pd.read_parquet(out_path).shape}")

Master shape: (970960, 107)
auto_not_cancel_ratio nulls: 0
plan_changes nulls: 0
Saved: /Users/harshithnr/kkbox-churn/data/parquet/master_dataset.parquet
Size: 202.9 MB
Shape verify: (970960, 107)


In [15]:
# Cell 11 — Full feature audit
feature_cols = [c for c in master.columns if c not in [
    'msno', 'is_churn', 
    'last_transaction_date', 'last_expire_date', 
    'last_listen_date', 'first_listen_date'
]]

print(f"Total features: {len(feature_cols)}")
print(f"\n--- TRANSACTION FEATURES ---")
txn_cols = [c for c in feature_cols if c in [
    'current_auto_renew','current_is_cancel','current_plan_days','current_payment_method',
    'days_since_last_transaction','days_since_last_cancel','customer_tenure_days','days_to_expiry',
    'total_transactions','total_cancellations','auto_renew_rate','cancel_rate',
    'total_amount_paid','avg_amount_paid','pay_per_day','avg_plan_days','max_plan_days',
    'has_long_plan','payment_methods_used','total_discount','discount_flag',
    'cancelled_last_30_days','cancel_last3','autorenew_last3','cancel_last5','autorenew_last5',
    'price_change','has_price_increase','payment_method_changed','auto_renew_switch',
    'switched_off_flag','avg_interval_days','nopay_ratio','t1_amount','t2_amount',
    'auto_not_cancel_ratio','plan_changes','last_churn','last_not_churn','last_not_in'
]]
print(f"Count: {len(txn_cols)}")
for c in txn_cols: print(f"  {c}")

print(f"\n--- MEMBERS FEATURES ---")
mem_cols = [c for c in feature_cols if c in [
    'city','registered_via','bd_clean','age_known','gender_known',
    'unknown_gender_young','account_age_days','is_city_1',
    'is_invalid_registration','registration_year','bd_bucket'
]]
print(f"Count: {len(mem_cols)}")
for c in mem_cols: print(f"  {c}")

print(f"\n--- LOG FEATURES ---")
log_cols = [c for c in feature_cols if c in [
    'days_since_last_listen','active_days_15d','active_days_30d','active_days_60d',
    'active_days_120d','active_days_all','has_listening_15d','has_listening_30d',
    'stopped_listening','listening_trend_ratio','active_days_trend',
    'completion_rate_15d','completion_rate_all','skip_rate_all',
    'secs_15d','secs_30d','secs_all','songs_15d','songs_30d','songs_all',
    'song_per_day_all','song_per_day_30d','song_per_day_15d','unq_per_day_all',
    'num_unq_15d','num_unq_30d','num_100_15d','num_100_30d','num_25_15d','num_25_30d',
    'listener_tenure_days','songs_std_30d','songs_std_15d','secs_std_30d','secs_std_15d',
    'final_day_songs','final_day_secs','final_day_completion_rate','final_day_skip_rate',
    'prev_active_days_15d','prev_secs_15d','prev_songs_15d'
]]
print(f"Count: {len(log_cols)}")
for c in log_cols: print(f"  {c}")

print(f"\n--- CROSS-TABLE FEATURES ---")
cross_cols = [c for c in feature_cols if c in [
    'has_listening_history','days_between_last_listen_and_expiry',
    'listening_per_dollar','is_extreme_risk','is_silent_loyal',
    'auto_off_inactive','listen_to_plan_ratio','active_days_per_sub_day'
]]
print(f"Count: {len(cross_cols)}")
for c in cross_cols: print(f"  {c}")

print(f"\n--- UNACCOUNTED COLUMNS ---")
accounted = set(txn_cols + mem_cols + log_cols + cross_cols)
unaccounted = [c for c in feature_cols if c not in accounted]
print(f"Count: {len(unaccounted)}")
for c in unaccounted: print(f"  {c}")

Total features: 101

--- TRANSACTION FEATURES ---
Count: 40
  current_auto_renew
  current_is_cancel
  current_plan_days
  current_payment_method
  days_since_last_transaction
  days_since_last_cancel
  customer_tenure_days
  days_to_expiry
  total_transactions
  total_cancellations
  auto_renew_rate
  cancel_rate
  total_amount_paid
  avg_amount_paid
  pay_per_day
  avg_plan_days
  max_plan_days
  has_long_plan
  payment_methods_used
  total_discount
  discount_flag
  cancelled_last_30_days
  cancel_last3
  autorenew_last3
  cancel_last5
  autorenew_last5
  price_change
  has_price_increase
  payment_method_changed
  auto_renew_switch
  switched_off_flag
  avg_interval_days
  nopay_ratio
  t1_amount
  t2_amount
  last_churn
  last_not_churn
  last_not_in
  auto_not_cancel_ratio
  plan_changes

--- MEMBERS FEATURES ---
Count: 11
  city
  registered_via
  bd_clean
  age_known
  gender_known
  unknown_gender_young
  account_age_days
  is_city_1
  is_invalid_registration
  registration_ye